<a href="https://colab.research.google.com/github/mohamedmorsy-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedmorsy-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:

# ML-05 — CELL 1
# Baseline rule and two signal checks

%pip -q install -U duckdb huggingface_hub pandas numpy

import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from getpass import getpass
from huggingface_hub import login
from IPython.display import display

print("=" * 60)
print("FLYRANK — BASELINE ACTION SCORING")
print("=" * 60)

print("Lane: Refresh / Content Opportunity Scoring")
print("Feature window: March 2026")
print("Evaluation window: April 2026")
print("Decision cutoff: 2026-03-31")

print("\nBASELINE RULE:")
print("Prioritize pages with at least 500 March impressions")
print("and at least 20% decline in daily impressions")
print("between the first and second halves of March.")

print("\nSINGLE REASON CODE: declining_with_demand")
print("ACTION: Review content refresh opportunity")

# Secure token input
token = getpass("\nEnter Hugging Face READ Token: ").strip()
assert token, "A Hugging Face token is required."

try:
    login(token=token, add_to_git_credential=False)
finally:
    del token

# DuckDB connection
con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    PROVIDER credential_chain
)
""")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{BASE}/fact_content_daily_performance"

MARCH = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{FACT}/month=2026-04/*.parquet')"

# March data only — future is excluded from features
print("\nLoading March data...")

march = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks
FROM {MARCH}
WHERE gsc_data_available IS TRUE
  AND report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""")

con.register("march_view", march)

# Aggregate at one row per client-content pair
features = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_march,
    SUM(gsc_clicks) AS clicks_march,
    COUNT(DISTINCT report_date) AS observed_days,

    SUM(CASE
        WHEN report_date < DATE '2026-03-16'
        THEN gsc_impressions ELSE 0 END
    ) AS early_impressions,

    SUM(CASE
        WHEN report_date >= DATE '2026-03-16'
        THEN gsc_impressions ELSE 0 END
    ) AS late_impressions,

    COUNT(DISTINCT CASE
        WHEN report_date < DATE '2026-03-16'
        THEN report_date END
    ) AS early_days,

    COUNT(DISTINCT CASE
        WHEN report_date >= DATE '2026-03-16'
        THEN report_date END
    ) AS late_days

FROM march_view
GROUP BY 1, 2
HAVING COUNT(DISTINCT report_date) >= 20
   AND SUM(gsc_impressions) >= 100
""").df()

# Normalize by observed days
features = features[
    (features["early_days"] >= 8) &
    (features["late_days"] >= 8) &
    (features["early_impressions"] > 0)
].copy()

features["early_daily"] = (
    features["early_impressions"] / features["early_days"]
)

features["late_daily"] = (
    features["late_impressions"] / features["late_days"]
)

features["march_drop_pct"] = (
    1 - features["late_daily"] / features["early_daily"]
)

features["ctr_march"] = (
    features["clicks_march"] /
    features["impressions_march"]
)

assert len(features) > 0, "No eligible March pages found."

print("March eligible pages:", len(features))

# April outcomes: evaluation ONLY
print("\nLoading April outcome data for signal audit...")

april = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_april,
    COUNT(DISTINCT report_date) AS april_days
FROM {APRIL}
WHERE gsc_data_available IS TRUE
  AND report_date >= DATE '2026-04-01'
  AND report_date < DATE '2026-05-01'
GROUP BY 1, 2
HAVING COUNT(DISTINCT report_date) >= 20
""").df()

audit = features.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

audit["future_decline"] = (
    audit["impressions_april"] / audit["april_days"]
    <= 0.80 * (
        audit["impressions_march"] /
        audit["observed_days"]
    )
).astype(int)

assert len(audit) > 0, "No valid April evaluation rows."

# One-word verdict based on observed differences
def verdict(high_rate, low_rate, high_n, low_n):
    if high_n < 30 or low_n < 30:
        return "MIXED"
    delta = high_rate - low_rate
    if delta >= 0.03:
        return "CONFIRMED"
    if delta <= -0.03:
        return "OPPOSITE"
    return "MIXED"

# SIGNAL 1: Search impression volume
audit["volume_bucket"] = pd.cut(
    audit["impressions_march"],
    bins=[99, 499, 1999, float("inf")],
    labels=["100-499", "500-1999", "2000+"]
)

volume_table = audit.groupby(
    "volume_bucket", observed=True
).agg(
    n=("future_decline", "size"),
    future_decline_rate=("future_decline", "mean"),
    median_impressions=("impressions_march", "median")
)

print("\nSIGNAL 1 — SEARCH VOLUME")
print("Flag-linked: Quick-win / demand threshold")
display(volume_table)

low = audit[audit["impressions_march"] < 500]
high = audit[audit["impressions_march"] >= 500]

volume_verdict = verdict(
    high["future_decline"].mean(),
    low["future_decline"].mean(),
    len(high),
    len(low)
)

print("VERDICT:", volume_verdict)
print("This tests risk association, not business impact.")

# SIGNAL 2: March performance trend
audit["trend_bucket"] = pd.cut(
    audit["march_drop_pct"],
    bins=[-float("inf"), -0.10, 0.10, 0.20, float("inf")],
    labels=["Growing", "Stable", "Mild decline", "Strong decline"]
)

trend_table = audit.groupby(
    "trend_bucket", observed=True
).agg(
    n=("future_decline", "size"),
    future_decline_rate=("future_decline", "mean")
)

print("\nSIGNAL 2 — RECENT PERFORMANCE TREND")
display(trend_table)

strong = audit[audit["march_drop_pct"] >= 0.20]
other = audit[audit["march_drop_pct"] < 0.20]

trend_verdict = verdict(
    strong["future_decline"].mean(),
    other["future_decline"].mean(),
    len(strong),
    len(other)
)

print("VERDICT:", trend_verdict)

print("\nBoth signal audits completed.")
print("April labels were used for evaluation only.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


FLYRANK — BASELINE ACTION SCORING
Lane: Refresh / Content Opportunity Scoring
Feature window: March 2026
Evaluation window: April 2026
Decision cutoff: 2026-03-31

BASELINE RULE:
Prioritize pages with at least 500 March impressions
and at least 20% decline in daily impressions
between the first and second halves of March.

SINGLE REASON CODE: declining_with_demand
ACTION: Review content refresh opportunity

Enter Hugging Face READ Token: ··········

Loading March data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March eligible pages: 89827

Loading April outcome data for signal audit...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


SIGNAL 1 — SEARCH VOLUME
Flag-linked: Quick-win / demand threshold


,n,future_decline_rate,median_impressions
volume_bucket,,,
100-499,28311,0.500442,245.0
500-1999,28222,0.545426,978.0
2000+,28680,0.464365,4836.5


VERDICT: MIXED
This tests risk association, not business impact.

SIGNAL 2 — RECENT PERFORMANCE TREND


,n,future_decline_rate
trend_bucket,,
Growing,35870,0.369166
Stable,15271,0.472137
Mild decline,7827,0.539670
Strong decline,26245,0.693580


VERDICT: CONFIRMED

Both signal audits completed.
April labels were used for evaluation only.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:

# ML-05 — CELL 2
# One rule, one reason code, one ranked queue

print("=" * 60)
print("BUILD BASELINE ACTION QUEUE")
print("=" * 60)

# The rule uses only March measurements
queue = features[
    (features["impressions_march"] >= 500) &
    (features["march_drop_pct"] >= 0.20)
].copy()

assert len(queue) > 0, "No pages match the baseline rule."

# Transparent baseline:
# Greater demand + stronger recent decline = higher priority
queue["baseline_score"] = (
    queue["impressions_march"] *
    queue["march_drop_pct"].clip(lower=0, upper=1)
).round(2)

# Exactly one positive reason code
queue["reason_code"] = "declining_with_demand"

# One associated action
queue["action_label"] = "Review content refresh opportunity"

# Sort and rank
queue = queue.sort_values(
    ["baseline_score", "impressions_march"],
    ascending=[False, False]
).reset_index(drop=True)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

# Only permitted March fields are exported
export_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions_march",
    "clicks_march",
    "ctr_march",
    "observed_days",
    "early_days",
    "late_days",
    "march_drop_pct",
    "baseline_score",
    "reason_code",
    "action_label"
]

# Create the required file
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "baseline_action_score.csv"

queue[export_columns].to_csv(csv_path, index=False)

print("\nBASELINE RESULTS")
print("Total eligible March pages:", len(features))
print("Ranked review candidates:", len(queue))
print("CSV saved to:", csv_path)

print("\nTop 10 scores:")
display(
    queue[
        ["rank", "impressions_march", "march_drop_pct",
         "baseline_score", "reason_code", "action_label"]
    ].head(10)
)

print("\nSUCCESS: Baseline ranked queue generated.")


BUILD BASELINE ACTION QUEUE

BASELINE RESULTS
Total eligible March pages: 89827
Ranked review candidates: 17488
CSV saved to: work/outputs/baseline_action_score.csv

Top 10 scores:


,rank,impressions_march,march_drop_pct,baseline_score,reason_code,action_label
0,1,83834.0,0.999306,83775.83,declining_with_demand,Review content refresh opportunity
1,2,244931.0,0.333686,81730.13,declining_with_demand,Review content refresh opportunity
2,3,245276.0,0.311971,76519.01,declining_with_demand,Review content refresh opportunity
3,4,132593.0,0.506393,67144.19,declining_with_demand,Review content refresh opportunity
4,5,64935.0,0.844171,54816.24,declining_with_demand,Review content refresh opportunity
5,6,194579.0,0.277783,54050.78,declining_with_demand,Review content refresh opportunity
6,7,140156.0,0.374022,52421.44,declining_with_demand,Review content refresh opportunity
7,8,65330.0,0.768176,50184.91,declining_with_demand,Review content refresh opportunity
8,9,57720.0,0.841795,48588.39,declining_with_demand,Review content refresh opportunity
9,10,58278.0,0.829587,48346.67,declining_with_demand,Review content refresh opportunity



SUCCESS: Baseline ranked queue generated.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:

# ML-05 — CELL 3
# Skeptical review of the top 10

print("=" * 60)
print("TOP-10 MANUAL REVIEW CHECKLIST")
print("=" * 60)

top10 = queue.head(10).copy()

reviews = []

for _, row in top10.iterrows():

    rank = int(row["rank"])
    impressions = int(row["impressions_march"])
    drop = float(row["march_drop_pct"])
    clicks = int(row["clicks_march"])

    action = "Review for content refresh"

    why = (
        f"{impressions:,} March impressions; "
        f"{drop:.1%} fall in daily impressions "
        f"between March halves."
    )

    # Skeptical reasons based on observable context
    if row["late_days"] < 12:
        weakness = (
            "Incomplete late-month coverage may "
            "exaggerate the apparent decline."
        )
    elif drop >= 0.85:
        weakness = (
            "A tracking interruption or migration "
            "could mimic a sharp traffic collapse."
        )
    elif clicks <= 5:
        weakness = (
            "High impressions with few clicks may "
            "reflect low relevance or search intent."
        )
    elif row["ctr_march"] < 0.005:
        weakness = (
            "Low CTR may be explained by search "
            "position or SERP layout."
        )
    else:
        weakness = (
            "Seasonality, competition, or a sibling "
            "page absorbing demand could explain it."
        )

    reviews.append({
        "rank": rank,
        "action": action,
        "why_selected": why,
        "what_would_make_it_wrong": weakness
    })

review_df = pd.DataFrame(reviews)

display(review_df)

print("\nTOP-10 REVIEW — ONE LINE PER CANDIDATE")

for r in reviews:
    print(
        f"\n#{r['rank']} | "
        f"Action: {r['action']} | "
        f"Why: {r['why_selected']} | "
        f"Could be wrong if: {r['what_would_make_it_wrong']}"
    )

print("\nReview all 10 explanations before acting.")
print("These are review candidates, not confirmed fixes.")


TOP-10 MANUAL REVIEW CHECKLIST


,rank,action,why_selected,what_would_make_it_wrong
0,1,Review for content refresh,"83,834 March impressions; 99.9% fall in daily ...",A tracking interruption or migration could mim...
1,2,Review for content refresh,"244,931 March impressions; 33.4% fall in daily...",Low CTR may be explained by search position or...
2,3,Review for content refresh,"245,276 March impressions; 31.2% fall in daily...","Seasonality, competition, or a sibling page ab..."
3,4,Review for content refresh,"132,593 March impressions; 50.6% fall in daily...",Low CTR may be explained by search position or...
4,5,Review for content refresh,"64,935 March impressions; 84.4% fall in daily ...",Low CTR may be explained by search position or...
5,6,Review for content refresh,"194,579 March impressions; 27.8% fall in daily...",Low CTR may be explained by search position or...
6,7,Review for content refresh,"140,156 March impressions; 37.4% fall in daily...",Low CTR may be explained by search position or...
7,8,Review for content refresh,"65,330 March impressions; 76.8% fall in daily ...",Low CTR may be explained by search position or...
8,9,Review for content refresh,"57,720 March impressions; 84.2% fall in daily ...",Low CTR may be explained by search position or...
9,10,Review for content refresh,"58,278 March impressions; 83.0% fall in daily ...",High impressions with few clicks may reflect l...



TOP-10 REVIEW — ONE LINE PER CANDIDATE

#1 | Action: Review for content refresh | Why: 83,834 March impressions; 99.9% fall in daily impressions between March halves. | Could be wrong if: A tracking interruption or migration could mimic a sharp traffic collapse.

#2 | Action: Review for content refresh | Why: 244,931 March impressions; 33.4% fall in daily impressions between March halves. | Could be wrong if: Low CTR may be explained by search position or SERP layout.

#3 | Action: Review for content refresh | Why: 245,276 March impressions; 31.2% fall in daily impressions between March halves. | Could be wrong if: Seasonality, competition, or a sibling page absorbing demand could explain it.

#4 | Action: Review for content refresh | Why: 132,593 March impressions; 50.6% fall in daily impressions between March halves. | Could be wrong if: Low CTR may be explained by search position or SERP layout.

#5 | Action: Review for content refresh | Why: 64,935 March impressions; 84.4% fall in

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:

# ML-05 — CELL 4
# Weak picks, leakage prevention, final checks

import json

print("=" * 60)
print("BASELINE QUALITY AND LEAKAGE AUDIT")
print("=" * 60)

# Weak candidates: low clicks, incomplete coverage,
# or extreme declines requiring manual investigation
weak_picks = queue[
    (queue["clicks_march"] <= 5) |
    (queue["late_days"] < 12) |
    (queue["march_drop_pct"] >= 0.85)
].copy()

print("\nPotential weak picks:", len(weak_picks))

print("\nExamples for manual investigation:")
display(
    weak_picks[
        ["rank", "impressions_march",
         "clicks_march", "late_days", "march_drop_pct"]
    ].head(10)
)

print("\nPotential explanations:")
print("- Seasonal loss of search demand")
print("- Tracking or reporting interruptions")
print("- Content consolidation or migration")
print("- Search intent or SERP changes")
print("- Very low click volume")

# Explicit score-input audit
score_inputs = {
    "impressions_march",
    "march_drop_pct"
}

forbidden_inputs = {
    "impressions_april",
    "april_days",
    "future_decline",
    "is_declining_label",
    "leaked_label_copy",
    "health_score",
    "priority_score",
    "recommended_action"
}

assert score_inputs.isdisjoint(forbidden_inputs)

# Ensure future outcomes never entered the ranked output
assert forbidden_inputs.isdisjoint(set(queue.columns))
assert forbidden_inputs.isdisjoint(set(export_columns))

# Final queue checks
assert len(queue) > 0
assert queue["baseline_score"].notna().all()
assert queue["baseline_score"].is_monotonic_decreasing
assert queue["reason_code"].nunique() == 1
assert queue["action_label"].nunique() == 1
assert len(review_df) == min(10, len(queue))
assert csv_path.exists()

# Produce a small metrics receipt, safe to commit
metrics = {
    "assignment": "Baseline Action Score",
    "lane": "Refresh / Content Opportunity Scoring",
    "feature_month": "2026-03",
    "evaluation_month": "2026-04",
    "baseline_rule": "March impressions >= 500 and within-March decline >= 20%",
    "signal_1_verdict": volume_verdict,
    "signal_2_verdict": trend_verdict,
    "total_eligible": int(len(features)),
    "ranked_candidates": int(len(queue)),
    "reviewed_top_k": int(len(review_df)),
    "future_inputs_used_for_score": False,
    "reason_code": "declining_with_demand",
    "csv_generated": True
}

metrics_path = output_dir / "baseline_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("\nFINAL CHECKLIST")
print("Two signal bucket tables: DONE")
print("Two signal verdicts: DONE")
print("One transparent scoring rule: DONE")
print("Single reason code: DONE")
print("Action labels: DONE")
print("Ranked CSV: DONE")
print("Top-10 review: DONE")
print("Leakage audit: PASSED")

print("\nCSV:", csv_path)
print("Metrics JSON:", metrics_path)

print("\nALL FINAL CHECKS PASSED")
print("Ready to review and save the notebook.")


BASELINE QUALITY AND LEAKAGE AUDIT

Potential weak picks: 11916

Examples for manual investigation:


,rank,impressions_march,clicks_march,late_days,march_drop_pct
0,1,83834.0,1.0,16,0.999306
9,10,58278.0,5.0,16,0.829587
12,13,97378.0,2.0,16,0.474911
26,27,47736.0,4.0,16,0.674589
48,49,51059.0,4.0,16,0.500718
52,53,37196.0,1.0,16,0.663804
58,59,38865.0,0.0,16,0.622835
59,60,39405.0,3.0,16,0.610172
62,63,25704.0,53.0,16,0.918535
68,69,33592.0,5.0,16,0.692688



Potential explanations:
- Seasonal loss of search demand
- Tracking or reporting interruptions
- Content consolidation or migration
- Search intent or SERP changes
- Very low click volume

FINAL CHECKLIST
Two signal bucket tables: DONE
Two signal verdicts: DONE
One transparent scoring rule: DONE
Single reason code: DONE
Action labels: DONE
Ranked CSV: DONE
Top-10 review: DONE
Leakage audit: PASSED

CSV: work/outputs/baseline_action_score.csv
Metrics JSON: work/outputs/baseline_metrics.json

ALL FINAL CHECKS PASSED
Ready to review and save the notebook.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.